## Now import ModelFlow

In [1]:
from modelclass import model

## Load the Pakistan model and run the baseline

In [2]:
mpak,bline = model.modelload('data/pak_new.pcim',run=True,keep='Baseline')

Open file from URL:  https://raw.githubusercontent.com/IbHansen/modelflow-manual/main/model_repo/pak_new.pcim


HTTPError: HTTP Error 404: Not Found

## Simulating the impact of imposing a carbon price

A simulation follows three steps:

1. **Create a scenario DataFrame** from a baseline using `.upd()`, which applies an update expression that sets, scales, or grows selected variables over a chosen time range.
2. **Solve the model** by calling it with the scenario DataFrame, a start and end year, and a `keep=` label that names the scenario for later comparison.
3. **Inspect results** on the returned DataFrame or directly on the model, which stores each kept scenario.

In [ ]:
CT30df = bline.upd("<2025 2050> PAKGGREVCO2CER PAKGGREVCO2GER PAKGGREVCO2OER = 30")

**Breakdown** 


$$
\begin{aligned}
&\underbrace{\texttt{CT30df}}_{\text{scenario df}}
 = \underbrace{\texttt{bline}}_{\text{baseline}}
 \,.\,\underbrace{\texttt{upd}}_{\text{accessor}}\bigl(\texttt{"}\ldots\texttt{"}\bigr) \\[8pt]
&\text{where the update string is} \\[4pt]
&\texttt{"}\,
 \overbrace{\texttt{<2025 2050>}}^{\text{years}}\ 
 \overbrace{\texttt{PAKGGREVCO2CER PAKGGREVCO2GER PAKGGREVCO2OER}}^{\text{coal, gas, oil CO}_2\text{ tax rates}}\ 
 \overbrace{\texttt{=}}^{\text{set}}\ 
 \overbrace{\texttt{30}}^{\text{USD/tCO}_2}
 \,\texttt{"}
\end{aligned}
$$

### Solve the model

In [ ]:
resultsdf = mpak(CT30df,2020,2050,keep="Nominal $30USD Carbon tax")

**Breakdown**
$$
\small
\underbrace{\texttt{resultsdf}}_{\text{results df}}
= \underbrace{\texttt{mpak}}_{\text{model}}\bigl(
\underbrace{\texttt{CT30df}}_{\text{scenario df}},\;
\underbrace{\texttt{2020}}_{\text{start}},\;
\underbrace{\texttt{2050}}_{\text{end}},\;
\underbrace{\texttt{keep="Nominal \$30USD Carbon tax"}}_{\text{scenario label}}
\bigr)
$$

### Readable labels for the emission variables

To make tables and plots easier to read, we attach short descriptions to each emission variable and merge them into `mpak.var_description`. These labels can then be used instead of the raw codes wherever the model displays results.

In [ ]:
var_descriptions = {
    'PAKCCEMISCO2CKN': 'Coal emissions, tCO2e',
    'PAKCCEMISCO2GKN': 'Natural Gas emissions, tCO2e',
    'PAKCCEMISCO2OKN': 'Crude Oil emissions, tCO2e',
    'PAKCCEMISCO2TKN': 'Total emissions, tCO2e',
}

mpak.var_description = mpak.var_description | var_descriptions

### Inspect results

 **Note**
 
 `resultsdf` is rarely used directly. The model object `mpak` keeps the baseline (`mpak.basedf`) and the latest run (`mpak.lastdf`) internally, so the following cells can compare and plot results directly from `mpak`.

In [ ]:
mpak['PAKNYGDPMKTPCN PAKFMLBLPOLYXN PAKCCEMISCO2?KN']

### Demo cue-sheet — causality dashboard (carbon tax → emissions → GDP)

Recording sequence for the Venice talk (each step triggers a ~1 s graph morph):

1. Start on GDP `PAKNYGDPMKTPCN`, **Up = 1**, Down = 0 → small graph.
2. Increase **Up: 1 → 2 → 3 → 4** (pause ~2 s each) — GDP's upstream drivers unfold.
3. Dropdown → `PAKCCEMISCO2TKN` (total emissions) — graph morphs to the emissions block.
4. Show the **Chart** and **Attribution** tabs — the $30-tax impact numbers.
5. Toggle **Node display: Name → +values → +Attribution**; flip **orientation V → H**.

_Record the browser tab (OBS / `Win+Alt+R`) → export MP4 (H.264/AAC) → embed in PowerPoint._

In [ ]:
mpak.PAKNYGDPMKTPCN

In [ ]:
mpak.modeldash('PAKNYGDPMKTPCN')

In [ ]:
mpak.modeldash('PAKCCEMISCO2TKN')

In [ ]:
mpak['PAKNYGDPMKTPCN PAKFMLBLPOLYXN PAKCCEMISCO2?KN'].rplot(samefig=1,datatype='difpctlevel')

In [ ]:
mpak['PAKCCEMISCO2?KN'].difpctlevel.rename().plot(title="Emissions impact of a $30 USD Carbon tax");

**Breakdown**

| Element | What it does |
|:---|---|
| `mpak` | The model object holding the baseline and the latest simulation |
| `['PAKCCEMISCO2*']` | Select all variables matching the pattern — here, CO₂ emissions |
| `.difpctlevel` | Percent difference between the latest simulation and the baseline |
| `.rename()` | Use the variable descriptions |
| `.df` | Return the underlying pandas DataFrame (rows = years, columns = variables) |
| `.T` | Transpose so variables become rows and years become columns |
| `.style.format("{:.2f}")` | Render as a styled table with values shown to two decimal places |